# Follow up batch split

In [1]:
import scanpy as sc
from matplotlib import pyplot as plt
import seaborn as sns
import pandas as pd
from tqdm import tqdm
from pprint import pprint

sc.set_figure_params(frameon=False, dpi=80, fontsize=12)

from scatlastb_utils.io import read_anndata
from scatlastb_utils.utils import dask_compute

from batch_analysis_utils import clean_slot_names

/hpc/scratch/project.hca_integration/miniforge3/envs/scanpy/lib/python3.11/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


In [2]:
adata = read_anndata(
    'data/pipeline/HLCAv1/batch_analysis/majority_voting/dataset~batch_correction/file_id~collect:batch_correction.zarr',
    X='X',
    obs='obs',
    var='var',
    obsm='obsm',
    uns='uns',
    obsp='obsp',
    dask_slots=['layers', 'X', 'obsp'],
    dask=True,
    backed=True,
)
columns = [col for col in adata.obs.columns if 'gene_score' not in col]
adata.obs = adata.obs[columns]
clean_slot_names(adata, replace_str='metrics:')

dask: True, backed: True
chunks: (200000, -1)
Read slot "X", store as "X"...
Read X as backed sparse dask array...
Read slot "obs", store as "obs"...
Read slot "var", store as "var"...
Read slot "obsm", store as "obsm"...


Read obsm slots as_dask=False: 100%|██████████| 14/14 [00:02<00:00,  5.24it/s]

Read slot "uns", store as "uns"...



Read uns slots as_dask=False: 100%|██████████| 97/97 [00:24<00:00,  3.89it/s] 

Read slot "obsp", store as "obsp"...



Read obsp slots as_dask=True: 100%|██████████| 14/14 [00:00<00:00, 153.89it/s]

shape: (584944, 27957)


In [3]:
adata_new_batch = read_anndata(
    'data/pipeline/HLCAv1/batch_analysis/collect/'
    'dataset~Reintegration_new_batch_covariate.zarr',
    X='X',
    obs='obs',
    var='var',
    obsm='obsm',
    uns='uns',
    obsp='obsp',
    dask_slots=['layers', 'X', 'obsp'],
    dask=True,
    backed=True,
)
columns = [col for col in adata_new_batch.obs.columns if 'gene_score' not in col]
adata_new_batch.obs = adata_new_batch.obs[columns]
clean_slot_names(adata_new_batch, replace_str='metrics:')

dask: True, backed: True
chunks: (200000, -1)
Read slot "X", store as "X"...
Read X as backed sparse dask array...
Read slot "obs", store as "obs"...
Read slot "var", store as "var"...
Read slot "obsm", store as "obsm"...


Read obsm slots as_dask=False: 100%|██████████| 6/6 [00:01<00:00,  5.94it/s]

Read slot "uns", store as "uns"...



Read uns slots as_dask=False: 100%|██████████| 35/35 [00:02<00:00, 16.11it/s]

Read slot "obsp", store as "obsp"...



Read obsp slots as_dask=True: 100%|██████████| 6/6 [00:00<00:00, 146.29it/s]

shape: (584944, 27957)


In [4]:
adata.obsm['']

KeyError: ''

## Graph dissimilarity

In [ ]:
import scatlastb_utils as atl

In [ ]:
integration1, integration2 = integrations

In [ ]:
tqdm._instances.clear()

In [ ]:
%%time
comp = f'{integration1}-vs-{integration2}'

adata.obs[[
    f'avg_distance_1:{comp}',
    f'avg_distance_2:{comp}',
    f'avg_difference:{comp}',
    f'avg_distance_diff:{comp}',
    f'spearman_correlation:{comp}',
]] = atl.metrics.compare_distances(
    adata,
    obsp_connectivities_1=f'connectivities--{integration1}',
    obsp_distances_1=f'distances--{integration1}',
    obsm_key_1=f'X_emb--{integration1}',
    obsp_connectivities_2=f'connectivities--{integration2}',
    obsp_distances_2=f'distances--{integration2}',
    obsm_key_2=f'X_emb--{integration2}',
    scale_distances=True,
    log_scale_diffs=False,
    n_jobs=5,
)

comp = f'{integration2}-vs-{integration1}'
adata.obs[[
    f'avg_distance_1:{comp}',
    f'avg_distance_2:{comp}',
    f'avg_difference:{comp}',
    f'avg_distance_diff:{comp}',
    f'spearman_correlation:{comp}',
]] = atl.metrics.compare_distances(
    adata,
    obsp_connectivities_1=f'connectivities--{integration2}',
    obsp_distances_1=f'distances--{integration2}',
    obsm_key_1=f'X_emb--{integration2}',
    obsp_connectivities_2=f'connectivities--{integration1}',
    obsp_distances_2=f'distances--{integration1}',
    obsm_key_2=f'X_emb--{integration1}',
    scale_distances=True,
    log_scale_diffs=False,
    n_jobs=5,
)

In [ ]:
atl.me.plot_ranked_distances(adata, integration1, integration2)
atl.me.plot_distances_scatter(adata, integration1, integration2, s=1, alpha=0.5)
atl.me.plot_distances_scatter(adata, integration2, integration1, s=1, alpha=0.5)

In [ ]:
def plot_graph_dissimilarity_umap(
    adata,
    integration1,
    integration2,
    metric,
    max_quantile=None,
    **kwargs,
):
    comparisons = [
        f'{integration1}-vs-{integration2}',
        f'{integration2}-vs-{integration1}'
    ]
    colors = [f'{metric}:{comparison}' for comparison in comparisons]
    ncols = len(comparisons)
    
    if max_quantile:
        kwargs['vmax'] = adata.obs[colors].quantile(max_quantile).tolist()
    
    sc.pl.embedding(
        adata,
        basis=f'X_umap--{integration1}',
        color=colors,
        ncols=ncols,
        wspace=0.2,
        **kwargs,
    )
 
    sc.pl.embedding(
        adata,
        basis=f'X_umap--{integration2}',
        color=colors,
        ncols=ncols,
        wspace=0.2,
        **kwargs,
    )

In [ ]:
plot_graph_dissimilarity_umap(
    adata,
    integration1,
    integration2,
    metric='avg_distance_diff',
    cmap='viridis',
    size=2,
    vmin=0,
    max_quantile=0.999,
    mask_obs=adata.obs.eval('qc_status == "failed"')
)